# `Segment` basics -- building OBC segments standalone & interior

This notebook is a short tour of the `regional_mom6.segment.Segment` class 

The key idea: a `Segment` only needs a small slice of a horizontal grid (and,
optionally, a matching `regional_mom6.Topo` for masking) to describe a single
straight, index-aligned MOM6 open boundary segment. It never needs the full
`regional_mom6.experiment.Experiment` -- so you can build and inspect segments
entirely on their own, without setting up a whole case first.

We'll use a small synthetic grid + flat bathymetry (with a strip of land
along the top) so this notebook runs instantly with no data download.

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings("ignore")

from regional_mom6.grid import Grid
from regional_mom6.topo import Topo
from regional_mom6.segment import Segment

## Step 1: A small synthetic grid + bathymetry

A 20x20 T-cell rectilinear domain, flat at 100 m depth, with a full-width strip
of land across the top 5 rows (rows 15-19) -- e.g. a coastline running along
the domain's north edge -- so we have some land to build boundaries around and
against.

In [ ]:
grid = Grid(
    resolution=1,
    xstart=0,
    lenx=20,
    ystart=0,
    leny=20,
    name="segment_demo_domain",
    type="rectilinear_cartesian",
)
topo = Topo(grid, min_depth=5.0, git=False)
topo.set_flat(100.0)

depth = topo.depth.values.copy()
depth[15:20, :] = 0.0  # land strip along the north edge, full width
topo.depth = depth

hgrid = grid._supergrid.to_ds(name=grid.name, author="segment-basics-demo")
hgrid

## Step 2: The easy case -- a full cardinal edge

`Segment.cardinal` is a convenience wrapper for the 4 full-edge cases
(`'north'`, `'south'`, `'east'`, `'west'`). Pass the `hgrid` and (optionally) a
`topo` to get land masking for free.

In [ ]:
south_segment = Segment.cardinal(hgrid, "south", "segment_001", topo=topo)

print(south_segment.mom6_obc_position_string())
south_segment.lon.values, south_segment.mask.values

## Step 3: An interior/arbitrary segment

For anything that isn't a full outer edge -- a partial edge, or a truly
interior line -- use `Segment.from_hgrid` directly.

A few things to keep in mind (see the module docstring in `segment.py` for the
full explanation):

- `axis="nyp"` gives a *horizontal* line (varies in x, sits on MOM6's V-point
  grid); `axis="nxp"` gives a *vertical* line (varies in y, U-point grid).
- `index` is a **supergrid** index (`2 * k + 1` for native T-row/column `k`),
  not a native grid index.
- `index_range` must resolve to an odd number of supergrid points
  (MOM6's BRUSHCUTTER_MODE requirement) -- for T-cell indices `lo`/`hi`, use
  `slice(2 * lo + 1, 2 * hi + 2)`.
- `ocean_side` says which compass side is the interior/ocean side -- `Segment`
  handles MOM6's own-position-masking asymmetry internally, so you never have
  to shift `index` yourself to compensate.
- an interior segment's two endpoints must be **land-capped** -- the T-cell
  one step past each end, on the land side, has to actually be land, or
  `from_hgrid` raises `ValueError` (a segment that ends flush against open
  water at a concave corner can blow MOM6 up within its first timestep). Below,
  T-row 14 sits just south of our land strip (which starts at row 15), and the
  land strip is full-width -- wider than the segment's own T-columns 7-15 --
  so both endpoints land safely under it.

Here we cut an interior line along native T-row 14 (supergrid index `29`),
spanning T-columns 7-15, with the ocean to the south.

In [ ]:
interior_segment = Segment.from_hgrid(
    hgrid,
    axis="nyp",
    index=29,  # native T-row 14 -> supergrid index 2*14+1
    index_range=slice(15, 32),  # native T-columns 7-15 -> slice(2*7+1, 2*15+2)
    segment_name="segment_002",
    topo=topo,
    ocean_side="south",
)

print(interior_segment.mom6_obc_position_string())
interior_segment.lon.values, interior_segment.mask.values

## Step 4: Building a segment from physical lon/lat instead of indices

`Segment.from_lonlat` resolves the nearest T-cell on the grid for you, so you
can think in physical coordinates instead of supergrid indices. It's exact on
a uniform rectilinear grid like this one.

This one cuts a *vertical* line at T-column 4, spanning T-rows 16-18 --
comfortably inside the land strip's row range (15-19), so both endpoints
land-cap against that same strip.

In [ ]:
lonlat_segment = Segment.from_lonlat(
    hgrid,
    axis="nxp",
    fixed_lon=4.5,
    lat_range=(16.5, 18.5),
    segment_name="segment_003",
    topo=topo,
    ocean_side="west",
)

print(lonlat_segment.mom6_obc_position_string())
lonlat_segment.lat.values, lonlat_segment.mask.values

## Step 5: Letting the bathymetry tell you which cardinal edges are open

`Segment.detect_open_cardinal_boundaries` looks at `topo.supergridmask` and
returns just the cardinal edges that actually touch ocean -- useful as a
sensible default when building a segment list for a case.
It only applies to the 4 cardinal edges -- custom/interior boundaries like
`segment_002` and `segment_003` above always have to be specified explicitly.

Our land strip covers the *entire* north edge, so `'north'` should be the one
cardinal direction missing from the result below.

In [ ]:
Segment.detect_open_cardinal_boundaries(topo)

## Where to go from here

Once you have a `Segment`, the next steps are:

- `segment.regrid_velocity_tracers(...)` -- regrid raw velocity/tracer forcing
  data onto the segment and write the MOM6-ready OBC segment file.
- `segment.regrid_tides(...)` -- same, for tidal boundary forcing.
- `segment.to_spec()` / `Segment.from_spec()` -- serialize a segment's
  axis/index/index_range/orientation to a small JSON-able dict and rebuild it
  later, without holding onto the `hgrid`/`topo` objects.

Both of those need real forcing data, so they're outside the scope of this
quick geometry-focused demo -- see `regional_mom6.experiment.Experiment`'s
OBC setup for a full end-to-end example.